# Lab MLflow Pipeline E2E — Jour 2

**Profil docker :** `mlops` · **Durée :** 75 min · **Format :** Trio rôlé

---

## Pipeline complet

```
Feast (offline store)          MLflow
       │                          │
       ▼                          ▼
  Features RFM  ──→  Entraînement  ──→  Experiment Tracking
       +                          ↓
  Labels churn   ──→  Model Registry  ──→  Serving API
                           │
                    Artefacts sur MinIO
                    (s3://mlflow/artifacts/)
```

## Contexte — Mission client

> RetailCo veut un modèle de churn **reproductible** : deux ingénieurs doivent
> pouvoir réentraîner le modèle 6 mois plus tard et obtenir exactement le même résultat.
>
> Votre mission : construire le pipeline d'entraînement avec traçabilité complète
> (features, paramètres, métriques, artefacts) via MLflow et Nessie.

In [ ]:
import os, json, subprocess, time
import numpy as np
import pandas as pd
from datetime import datetime
import requests

# mlflow, scikit-learn, feast sont dans requirements.txt
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

# Config MLflow
os.environ['MLFLOW_TRACKING_URI']  = 'http://mlflow:5000'
os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'http://minio:9000'
os.environ['AWS_ACCESS_KEY_ID']    = 'minioadmin'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'minioadmin'
mlflow.set_tracking_uri('http://mlflow:5000')

NESSIE_API = 'http://nessie:19120/api/v2'

def nessie(path):
    r = requests.get(f'{NESSIE_API}{path}')
    try:
        return r.json()
    except Exception:
        return {}

print(f'✅ MLflow tracking URI : {mlflow.get_tracking_uri()}')
print('✅ Artefacts stockés sur MinIO (s3://mlflow/artifacts/)')

# Vérifier connexion MLflow
try:
    experiments = mlflow.search_experiments()
    print(f'✅ MLflow connecté : {len(experiments)} expérience(s)')
except Exception as e:
    print(f'⚠️  MLflow: {e}')

---
## ⏱️ Temps 1 — Kata d'amorce (10 min)

Créer une expérience MLflow et logger un run de test.

In [ ]:
# ── Kata : premier run MLflow ─────────────────────────────────────────────────
mlflow.set_experiment('RetailCo-ChurnPrediction')

with mlflow.start_run(run_name='kata-test') as run:
    mlflow.log_param('model_type', 'GradientBoosting')
    mlflow.log_param('n_estimators', 10)
    mlflow.log_metric('test_auc', 0.75)
    mlflow.set_tag('purpose', 'kata-amorce')
    run_id = run.info.run_id

print(f'✅ Run MLflow créé : {run_id}')
print(f'   Voir : http://localhost:5000/#/experiments/1/runs/{run_id}')
print('\n📋 Artefacts dans MLflow UI :')
client = mlflow.tracking.MlflowClient()
run_info = client.get_run(run_id)
for k, v in run_info.data.params.items():
    print(f'   param.{k} = {v}')
for k, v in run_info.data.metrics.items():
    print(f'   metric.{k} = {v}')

---
## ⏱️ Temps 2 — Pipeline complet avec traçabilité (45 min)

### Partie A — Préparer les features (10 min)

In [ ]:
# ── Générer features + labels ─────────────────────────────────────────────────
np.random.seed(42)
n = 2000

# Features (normalement issues de Feast offline store)
X_raw = pd.DataFrame({
    'customer_id':    [f'cust-{i:04d}' for i in range(n)],
    'recency_days':   np.random.randint(1, 365, n),
    'frequency_30d':  np.random.randint(0, 15, n),
    'monetary_90d':   np.round(np.random.exponential(50000, n), 2),
    'avg_basket_size':np.round(np.random.uniform(5000, 200000, n), 2),
    'return_rate':    np.round(np.random.beta(2, 8, n), 3),
})

# Labels churn (simulés)
churn_prob = (
    0.3 * (X_raw['recency_days'] > 90).astype(float)
  + 0.3 * (X_raw['frequency_30d'] < 2).astype(float)
  + 0.2 * (X_raw['return_rate'] > 0.15).astype(float)
  + 0.2 * np.random.random(n)
)
y = (churn_prob > 0.5).astype(int)

FEATURES = ['recency_days', 'frequency_30d', 'monetary_90d', 'avg_basket_size', 'return_rate']
X = X_raw[FEATURES]

print(f'✅ Dataset : {len(X)} clients')
print(f'   Churn rate : {y.mean():.1%}')
print(f'   Features   : {FEATURES}')

# Récupérer le hash Nessie (pour la traçabilité)
nessie_hash = nessie('/trees/main').get('reference', {}).get('hash', 'unknown')[:16]
print(f'\n📌 Nessie hash des données : {nessie_hash}...')

### Partie B — Entraînement avec traçabilité complète (20 min)

**TODO 1** : Entraînez le modèle de churn en loggant dans MLflow :
- Le hash Nessie des données utilisées
- Les paramètres du modèle
- Les métriques sur le jeu de test
- Le modèle lui-même (artefact sur MinIO)
- La feature importance

In [ ]:
# TODO 1 — Pipeline d'entraînement avec MLflow
# 📝 Complétez le pipeline avec les bons logs MLflow

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# TODO : Démarrer un run MLflow et logger :
# - nessie_hash comme tag
# - les paramètres du GradientBoostingClassifier
# - AUC, precision, recall, F1 sur le test set
# - le modèle avec mlflow.sklearn.log_model
# - la feature importance

PARAMS = {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1, 'random_state': 42}

print('💡 mlflow.start_run(run_name="churn-v1")')
print('💡 mlflow.log_params(PARAMS)')
print('💡 mlflow.set_tag("nessie_data_hash", nessie_hash)')
print('💡 mlflow.log_metric("test_auc", roc_auc_score(y_test, y_pred_proba))')
print('💡 mlflow.sklearn.log_model(model, "model")')

In [ ]:
# ✅ SOLUTION TODO 1

# Configuration MinIO pour MLflow artifacts
import os
os.environ.setdefault('MLFLOW_S3_ENDPOINT_URL', 'http://minio:9000')
os.environ.setdefault('AWS_ACCESS_KEY_ID', 'minioadmin')
os.environ.setdefault('AWS_SECRET_ACCESS_KEY', 'minioadmin')
mlflow.set_experiment('RetailCo-ChurnPrediction')

with mlflow.start_run(run_name='churn-gradient-boosting-v1') as run:
    # Tags de traçabilité
    mlflow.set_tags({
        'nessie_data_hash':   nessie_hash,
        'data_version':       '1.0',
        'feature_set':        'customer_rfm_v1',
        'environment':        'formation',
        'team':               'data-science',
    })

    # Paramètres
    mlflow.log_params(PARAMS)
    mlflow.log_params({
        'test_size': 0.2,
        'scaler': 'StandardScaler',
        'n_features': len(FEATURES),
        'n_train': len(X_train),
        'n_test': len(X_test),
        'train_churn_rate': round(y_train.mean(), 4),
    })

    # Entraînement
    model = GradientBoostingClassifier(**PARAMS)
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]

    # Métriques
    metrics = {
        'test_auc':       round(roc_auc_score(y_test, y_proba), 4),
        'test_precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'test_recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'test_f1':        round(f1_score(y_test, y_pred, zero_division=0), 4),
        'train_auc':      round(roc_auc_score(y_train, model.predict_proba(X_train_s)[:, 1]), 4),
    }
    mlflow.log_metrics(metrics)

    # Feature importance
    for feat, imp in zip(FEATURES, model.feature_importances_):
        mlflow.log_metric(f'importance_{feat}', round(float(imp), 4))

    # Modèle
    mlflow.sklearn.log_model(
        model, name='model',
        registered_model_name='RetailCoChurnModel',
        input_example=X_test_s[:3],
    )

    run_id = run.info.run_id

print('✅ Run MLflow terminé :')
for k, v in metrics.items():
    print(f'   {k} = {v}')
print(f'\n🔗 MLflow UI : http://localhost:5000/#/experiments/1/runs/{run_id}')
print(f'📦 Artefacts : s3://mlflow/artifacts/{run_id}/model/')

### Partie C — Tester la reproductibilité (15 min)

**TODO 2** : Chargez le modèle depuis MLflow et vérifiez que vous obtenez exactement les mêmes prédictions qu'au moment de l'entraînement.

In [ ]:
# TODO 2 — Reproductibilité
# 📝 a) Charger le modèle depuis MLflow Model Registry
#    b) Faire des prédictions sur X_test_s
#    c) Vérifier que les métriques sont identiques
#    d) Afficher le tag nessie_data_hash pour montrer la traçabilité

print('💡 mlflow.sklearn.load_model(f"models:/RetailCoChurnModel/1")')
print('💡 Comparer avec les métriques loggées dans le run')

In [ ]:
# ✅ SOLUTION TODO 2

# Charger le modèle depuis le Model Registry
# Note : si ce notebook est relancé plusieurs fois, de nouvelles versions du modèle
# s'accumulent dans le Model Registry (version 1, 2, 3...). On charge ici la
# version 1 spécifiquement pour la démo — en pratique on utiliserait souvent
# un alias comme 'champion' ou 'latest' plutôt qu'un numéro de version fixe.
try:
    loaded_model = mlflow.sklearn.load_model('models:/RetailCoChurnModel/1')
    y_pred_loaded = loaded_model.predict(X_test_s)
    y_proba_loaded = loaded_model.predict_proba(X_test_s)[:, 1]

    repro_auc = round(roc_auc_score(y_test, y_proba_loaded), 4)
    print(f'✅ AUC modèle rechargé : {repro_auc}')
    print(f'   AUC original        : {metrics["test_auc"]}')
    print(f'   Identique           : {repro_auc == metrics["test_auc"]}')

    # Afficher la traçabilité
    client = mlflow.tracking.MlflowClient()
    run_data = client.get_run(run_id)
    print(f'\n📌 Traçabilité du modèle :')
    print(f'   Nessie data hash : {run_data.data.tags.get("nessie_data_hash")}')
    print(f'   Feature set      : {run_data.data.tags.get("feature_set")}')
    print(f'   Run ID           : {run_id}')
    print('\n✅ Pour reproduire ce modèle dans 6 mois :')
    print(f'   1. Charger le hash Nessie : {run_data.data.tags.get("nessie_data_hash")}')
    print('   2. Pointer Spark sur ce hash → mêmes données Silver')
    print('   3. Relancer ce notebook → mêmes features → même modèle')
except Exception as e:
    print(f'Model Registry: {e}')
    print('💡 Le modèle est peut-être en cours d\'enregistrement — relancer dans 30s')

---
## ⏱️ Temps 3 — Extension (10 min)

Comparer deux versions du modèle : avec et sans la feature `return_rate`.

In [ ]:
# ── Comparaison deux expériences ─────────────────────────────────────────────
results = []
for features_subset, name in [
    (FEATURES, 'all-features'),
    ([f for f in FEATURES if f != 'return_rate'], 'no-return-rate')
]:
    X_sub = X[features_subset]
    X_tr, X_te, y_tr, y_te = train_test_split(X_sub, y, test_size=0.2, random_state=42)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)

    with mlflow.start_run(run_name=f'churn-{name}'):
        m = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
        m.fit(X_tr_s, y_tr)
        auc = roc_auc_score(y_te, m.predict_proba(X_te_s)[:, 1])
        mlflow.log_param('feature_set', name)
        mlflow.log_metric('test_auc', round(auc, 4))
        results.append((name, round(auc, 4)))

print('📊 Comparaison des expériences :')
for name, auc in results:
    print(f'   {name:<20} : AUC = {auc}')
delta = results[0][1] - results[1][1]
print(f'\n   Impact de return_rate : {delta:+.4f} AUC')

---
## ⏱️ Débrief client (10 min)

### Questions — rôle Client :

1. **"Le hash Nessie dans MLflow — comment ça m'aide concrètement si mon modèle performe moins bien dans 3 mois ?"**
2. **"Les artefacts sont sur MinIO. Que se passe-t-il si MinIO tombe ? Mon modèle est-il perdu ?"**
3. **"On a loggé beaucoup de choses. En production avec 50 modèles actifs, comment on gère ça ?"**
4. **"Feast + MLflow + Nessie : est-ce qu'on a vraiment besoin des trois ? Lequel on enlève si on doit simplifier ?"**

---

## Récapitulatif

| Concept | Ce qu'on a fait | Impact |
|---------|----------------|--------|
| Experiment tracking | Log params, métriques, tags | Comparer les runs facilement |
| Nessie hash dans MLflow | Tag `nessie_data_hash` | Reproduire exactement les données |
| Model Registry | `models:/RetailCoChurnModel/1` | Déploiement stable avec versioning |
| Artefacts sur MinIO | `s3://mlflow/artifacts/` | Stockage décentralisé, pas de lock-in |
| Reproductibilité | Charger + re-inférer | Même résultat 6 mois plus tard |